# Text-to-music retrieval model benchmark

Quick-and-dirty inference benchmark for **text → music retrieval** models, aimed at picking a
model small + fast enough for on-device iOS use (paired with quantization, similar in spirit to
`notebooks/benchmark_clip_models.ipynb`).

The task: encode a small library of music tracks, encode a text query (e.g. "a relaxing acoustic
guitar tune"), and retrieve the most relevant track(s) by cosine similarity in a shared embedding
space.

## Model selection note

The original candidate list was **MERT, CLAP, MERIT, MuQ**. Two of those don't fit this task:

- **MERT** (`m-a-p/MERT-v1-*`) is an audio-only self-supervised encoder — it has no text tower, so
  it cannot score a text query against audio out of the box.
- **MERIT** sits on a *frozen MERT* backbone and is built for audio-to-audio similarity (melody /
  timbre / rhythm), not text-to-audio retrieval — also no text tower.

Both were swapped for real text→music/audio retrieval models (dual-tower, trained with a
contrastive text-audio objective) so all four candidates actually support the flow above:

| Slot | Model | Why |
|---|---|---|
| CLAP | `laion/larger_clap_music_and_speech` | LAION CLAP, music+speech-finetuned checkpoint (HTSAT audio + RoBERTa text) |
| MuQ | `OpenMuQ/MuQ-MuLan-large` | Tencent MuQ-MuLan (Conformer audio + XLM-RoBERTa text) |
| MERT replacement | `laion/clap-htsat-unfused` | Original general-audio LAION CLAP checkpoint — same family, different training data/size, so it's a useful size/quality data point |
| MERIT replacement | Wav2CLIP + CLIP text tower | `wav2clip` audio encoder (distilled from CLIP) paired with `openai/clip-vit-base-patch32`'s text tower — a much smaller, older architecture, useful as a lower bound |

(Microsoft's MS-CLAP was also considered but its PyPI package hard-pins `numpy<2.0`, which conflicts
with `numpy 2.5` already required elsewhere in this shared `trial_models` environment — skipped to
avoid destabilizing the shared `uv` environment used by the other notebooks here.)

One more finding worth flagging: the `laion/larger_clap_music` checkpoint (the "obvious" music-CLAP
choice) produces **collapsed embeddings** when loaded through `transformers==4.51.3` — every track's
audio embedding is >0.98 cosine-similar to every other track's, and every text embedding is >0.998
similar to every other text's, regardless of content, so text→audio similarity is just noise
(confirmed by comparing audio-audio and text-text similarity matrices in isolation, not just the
retrieval output). `laion/larger_clap_music_and_speech` and `laion/larger_clap_general` do **not**
have this problem and were used instead.

**Env setup (uv), run from `trial_models/`:**
```bash
uv sync
uv run --with jupyter jupyter lab music_retrieval_comparison/compare_music_retrieval_models.ipynb
```
or select the `.venv` created by `uv sync` as this notebook's kernel.

In [1]:
import io
import statistics
import time
import warnings
from pathlib import Path

import librosa
import torch

warnings.filterwarnings("ignore")
torch.backends.quantized.engine = "qnnpack"  # dynamic INT8 quantization backend available on Apple Silicon

In [2]:
AUDIO_DIR = Path.cwd().parent / "audios"
CLIP_SECONDS = 30  # truncate each track to its first 30s (keeps encoding fast + comparable across models)
QUERY_TEXTS = [
    "an upbeat, danceable pop song",
    "a slow, emotional piano ballad",
    "a relaxing acoustic guitar tune",
    "a melancholic, introspective song",
    "a warm, romantic jazz standard",
    "a high-energy rock anthem",
]
WARMUP_RUNS = 1
TIMED_RUNS = 3

## Helpers

`load_audio_files(sr)` decodes every `.mp3` in `../audios` at a given model's target sample rate.
`time_it` times a full batch pass over all tracks (mirrors `benchmark_clip_models.ipynb`).
`state_dict_size_mb` gives an actual on-disk-equivalent size (not just `params * 4 bytes`), so INT8
dynamic quantization shows up correctly even though only `Linear`/`Conv1d` layers get quantized.

In [3]:
def load_audio_files(sr):
    paths = sorted(p for p in AUDIO_DIR.iterdir() if p.suffix.lower() == ".mp3")
    if not paths:
        raise FileNotFoundError(f"No audio files found in {AUDIO_DIR}")
    n_samples = sr * CLIP_SECONDS
    waves = [
        librosa.load(p, sr=sr, mono=True, duration=CLIP_SECONDS)[0][:n_samples] for p in paths
    ]
    return paths, waves


def time_it(fn, warmup=WARMUP_RUNS, runs=TIMED_RUNS):
    for _ in range(warmup):
        fn()
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        fn()
        times.append((time.perf_counter() - t0) * 1000)
    return statistics.mean(times), (statistics.stdev(times) if len(times) > 1 else 0.0)


def state_dict_size_mb(module):
    buf = io.BytesIO()
    torch.save(module.state_dict(), buf)
    return buf.getbuffer().nbytes / 1e6


def show_retrieval(track_paths, track_embeds, query_texts, text_embeds, top_k=3):
    sims = text_embeds @ track_embeds.T  # [num_queries, num_tracks]
    for qi, q in enumerate(query_texts):
        order = torch.argsort(sims[qi], descending=True)[:top_k]
        matches = ", ".join(
            f"{track_paths[idx].stem} ({sims[qi, idx]:.3f})" for idx in order
        )
        print(f'    "{q}" -> {matches}')

## Model adapters

Each loader returns `(encode_audio_fn, encode_text_fn, num_params, size_modules, quantize_fn, sample_rate)`.
- `encode_audio_fn(waveforms)` -> L2-normalized tensor `[N, D]`
- `encode_text_fn(texts)` -> L2-normalized tensor `[M, D]`
- `size_modules` are the `nn.Module`s used for on-disk size accounting
- `quantize_fn()` mutates those modules **in place** to dynamic INT8 (CPU-only, `qnnpack` backend)

In [4]:
def load_clap(model_id):
    from transformers import ClapModel, ClapProcessor

    model = ClapModel.from_pretrained(model_id)
    processor = ClapProcessor.from_pretrained(model_id)
    model.eval()
    n_params = sum(p.numel() for p in model.parameters())
    sample_rate = 48_000

    def encode_audio(waves):
        inputs = processor(audios=waves, sampling_rate=sample_rate, return_tensors="pt")
        with torch.no_grad():
            feats = model.get_audio_features(**inputs)
            feats = feats / feats.norm(dim=-1, keepdim=True)
        return feats

    def encode_text(texts):
        inputs = processor(text=texts, return_tensors="pt", padding=True)
        with torch.no_grad():
            feats = model.get_text_features(**inputs)
            feats = feats / feats.norm(dim=-1, keepdim=True)
        return feats

    def quantize():
        torch.quantization.quantize_dynamic(
            model, {torch.nn.Linear}, dtype=torch.qint8, inplace=True
        )

    return encode_audio, encode_text, n_params, [model], quantize, sample_rate

In [5]:
def load_muq_mulan(model_id="OpenMuQ/MuQ-MuLan-large"):
    from muq import MuQMuLan

    mulan = MuQMuLan.from_pretrained(model_id)
    mulan.eval()
    n_params = sum(p.numel() for p in mulan.parameters())
    sample_rate = 24_000

    def encode_audio(waves):
        wavs = torch.stack([torch.tensor(w) for w in waves])
        with torch.no_grad():
            feats = mulan(wavs=wavs)
            feats = feats / feats.norm(dim=-1, keepdim=True)
        return feats

    def encode_text(texts):
        with torch.no_grad():
            feats = mulan(texts=texts)
            feats = feats / feats.norm(dim=-1, keepdim=True)
        return feats

    def quantize():
        # inplace=True avoids a deepcopy that chokes on MuQ's weight-normed conv layers
        torch.quantization.quantize_dynamic(
            mulan, {torch.nn.Linear}, dtype=torch.qint8, inplace=True
        )

    return encode_audio, encode_text, n_params, [mulan], quantize, sample_rate

In [6]:
def load_wav2clip(text_model_id="openai/clip-vit-base-patch32"):
    import wav2clip
    from transformers import CLIPModel, CLIPTokenizer

    audio_model = wav2clip.get_model()
    text_model = CLIPModel.from_pretrained(text_model_id)
    tokenizer = CLIPTokenizer.from_pretrained(text_model_id)
    text_model.eval()
    n_params = sum(p.numel() for p in audio_model.parameters()) + sum(
        p.numel() for p in text_model.parameters()
    )
    sample_rate = 16_000

    def encode_audio(waves):
        import numpy as np

        batch = np.stack(waves)
        feats = torch.tensor(wav2clip.embed_audio(batch, audio_model))
        return feats / feats.norm(dim=-1, keepdim=True)

    def encode_text(texts):
        inputs = tokenizer(texts, return_tensors="pt", padding=True)
        with torch.no_grad():
            feats = text_model.get_text_features(**inputs)
        return feats / feats.norm(dim=-1, keepdim=True)

    def quantize():
        torch.quantization.quantize_dynamic(
            audio_model, {torch.nn.Linear, torch.nn.Conv1d}, dtype=torch.qint8, inplace=True
        )
        torch.quantization.quantize_dynamic(
            text_model, {torch.nn.Linear}, dtype=torch.qint8, inplace=True
        )

    return encode_audio, encode_text, n_params, [audio_model, text_model], quantize, sample_rate

## Candidate models

All four are genuine dual-tower text↔audio retrieval models (see the model-selection note above
for why MERT/MERIT were swapped out).

In [7]:
MODEL_SPECS = [
    {
        "label": "CLAP (LAION, music+speech-finetuned)",
        "loader": lambda: load_clap("laion/larger_clap_music_and_speech"),
    },
    {
        "label": "CLAP (LAION, general-audio, HTSAT-unfused)",
        "loader": lambda: load_clap("laion/clap-htsat-unfused"),
    },
    {
        "label": "MuQ-MuLan-large (Tencent)",
        "loader": load_muq_mulan,
    },
    {
        "label": "Wav2CLIP + CLIP ViT-B/32 text",
        "loader": load_wav2clip,
    },
]

## Benchmark

For each model: load, report params + FP32 size, time audio/text encoding, show FP32 text→track
retrieval, then apply dynamic INT8 quantization in place and re-time + re-check how many queries'
top-1 match changed.

In [8]:
def run_benchmark(spec):
    print(f"=== {spec['label']} ===")
    t0 = time.perf_counter()
    try:
        encode_audio, encode_text, n_params, size_modules, quantize, sr = spec["loader"]()
    except Exception as e:
        print(f"  FAILED to load: {e}\n")
        return None
    load_s = time.perf_counter() - t0

    track_paths, waves = load_audio_files(sr)
    fp32_mb = sum(state_dict_size_mb(m) for m in size_modules)
    print(
        f"  params: {n_params / 1e6:.1f}M | fp32 size: {fp32_mb:.1f}MB | load time: {load_s:.1f}s"
    )

    audio_mean, audio_std = time_it(lambda: encode_audio(waves))
    ms_per_track_fp32 = audio_mean / len(waves)
    text_mean, text_std = time_it(lambda: encode_text(QUERY_TEXTS))
    print(
        f"  [fp32] audio encode: {audio_mean:.0f}ms +/- {audio_std:.0f}ms "
        f"for {len(waves)} tracks ({ms_per_track_fp32:.0f}ms/track)"
    )
    print(
        f"  [fp32] text encode:  {text_mean:.0f}ms +/- {text_std:.0f}ms for {len(QUERY_TEXTS)} queries"
    )

    track_embeds = encode_audio(waves)
    text_embeds = encode_text(QUERY_TEXTS)
    embed_dim = track_embeds.shape[-1]
    print("  [fp32] retrieval (top-3 tracks per query):")
    show_retrieval(track_paths, track_embeds, QUERY_TEXTS, text_embeds)

    quantize()
    int8_mb = sum(state_dict_size_mb(m) for m in size_modules)

    audio_mean_q, audio_std_q = time_it(lambda: encode_audio(waves))
    ms_per_track_int8 = audio_mean_q / len(waves)
    text_mean_q, text_std_q = time_it(lambda: encode_text(QUERY_TEXTS))
    print(
        f"  [int8] audio encode: {audio_mean_q:.0f}ms +/- {audio_std_q:.0f}ms "
        f"({ms_per_track_int8:.0f}ms/track) | size: {int8_mb:.1f}MB"
    )
    print(
        f"  [int8] text encode:  {text_mean_q:.0f}ms +/- {text_std_q:.0f}ms for {len(QUERY_TEXTS)} queries"
    )

    q_track_embeds = encode_audio(waves)
    q_text_embeds = encode_text(QUERY_TEXTS)
    fp32_top1 = (text_embeds @ track_embeds.T).argmax(dim=-1)
    int8_top1 = (q_text_embeds @ q_track_embeds.T).argmax(dim=-1)
    top1_shift = int((fp32_top1 != int8_top1).sum())
    print(
        f"  [int8] top-1 retrieval changed for {top1_shift}/{len(QUERY_TEXTS)} queries after quantization\n"
    )

    return {
        "label": spec["label"],
        "params_m": n_params / 1e6,
        "embed_dim": embed_dim,
        "fp32_mb": fp32_mb,
        "int8_mb": int8_mb,
        "load_s": load_s,
        "ms_per_track_fp32": ms_per_track_fp32,
        "ms_per_track_int8": ms_per_track_int8,
        "ms_per_text_batch_fp32": text_mean,
        "ms_per_text_batch_int8": text_mean_q,
        "top1_shift": top1_shift,
    }

In [9]:
results = [r for r in (run_benchmark(spec) for spec in MODEL_SPECS) if r is not None]

=== CLAP (LAION, music+speech-finetuned) ===


  params: 193.9M | fp32 size: 776.5MB | load time: 3.6s


  [fp32] audio encode: 414ms +/- 16ms for 9 tracks (46ms/track)
  [fp32] text encode:  19ms +/- 0ms for 6 queries


  [fp32] retrieval (top-3 tracks per query):
    "an upbeat, danceable pop song" -> I Hope to Be Around (0.415), Clementine (0.403), Happy! (0.382)
    "a slow, emotional piano ballad" -> Regent's Park (0.303), I Hope to Be Around (0.281), Clementine (0.272)
    "a relaxing acoustic guitar tune" -> The Most Beautiful Thing (0.387), I Hope to Be Around (0.369), Regent's Park (0.283)
    "a melancholic, introspective song" -> Clementine (0.539), Regent's Park (0.495), I Hope to Be Around (0.478)
    "a warm, romantic jazz standard" -> Cheek To Cheek (0.466), Cherry Wine (0.460), From The Start (0.439)
    "a high-energy rock anthem" -> Happy! (0.366), Clementine (0.327), From The Start (0.323)


[W908 13:30:02.634643000 qlinear_dynamic.cpp:251] Warning: Currently, qnnpack incorrectly ignores reduce_range when it is set to true; this may change in a future release. (function operator())


...

  params: 163.0M | fp32 size: 652.1MB | load time: 1.4s


  [fp32] audio encode: 338ms +/- 63ms for 9 tracks (38ms/track)
  [fp32] text encode:  9ms +/- 0ms for 6 queries


  [fp32] retrieval (top-3 tracks per query):
    "an upbeat, danceable pop song" -> Regent's Park (0.377), I Hope to Be Around (0.357), Happy! (0.337)
    "a slow, emotional piano ballad" -> Regent's Park (0.375), I Hope to Be Around (0.361), Clementine (0.314)
    "a relaxing acoustic guitar tune" -> Regent's Park (0.381), From The Start (0.363), The Most Beautiful Thing (0.357)
    "a melancholic, introspective song" -> Regent's Park (0.375), I Hope to Be Around (0.349), Clementine (0.312)
    "a warm, romantic jazz standard" -> Regent's Park (0.416), I Hope to Be Around (0.361), Happy! (0.356)
    "a high-energy rock anthem" -> Regent's Park (0.359), I Hope to Be Around (0.345), Happy! (0.323)


  [int8] audio encode: 308ms +/- 10ms (34ms/track) | size: 280.7MB
  [int8] text encode:  17ms +/- 1ms for 6 queries


  [int8] top-1 retrieval changed for 0/6 queries after quantization



## Summary

In [10]:
header = (
    f"{'Model':42s} {'Params(M)':>10s} {'FP32(MB)':>9s} {'INT8(MB)':>9s} "
    f"{'Load(s)':>8s} {'ms/track(fp32->int8)':>22s} {'ms/text-batch(fp32->int8)':>26s} {'Dim':>5s}"
)
print(header)
for r in results:
    ms_track = f"{r['ms_per_track_fp32']:.0f}->{r['ms_per_track_int8']:.0f}"
    ms_text = f"{r['ms_per_text_batch_fp32']:.0f}->{r['ms_per_text_batch_int8']:.0f}"
    print(
        f"{r['label']:42s} {r['params_m']:10.1f} {r['fp32_mb']:9.1f} {r['int8_mb']:9.1f} "
        f"{r['load_s']:8.1f} {ms_track:>22s} {ms_text:>26s} {r['embed_dim']:5d}"
    )

Model                                       Params(M)  FP32(MB)  INT8(MB)  Load(s)   ms/track(fp32->int8)  ms/text-batch(fp32->int8)   Dim
CLAP (LAION, music+speech-finetuned)            193.9     776.5     312.8      3.6                 46->94                     19->30   512
CLAP (LAION, general-audio, HTSAT-unfused)      153.5     614.6     271.9      2.4                 32->49                     22->31   512
MuQ-MuLan-large (Tencent)                       663.4    2654.0    1411.6      6.8             1691->2393                     36->60   512
Wav2CLIP + CLIP ViT-B/32 text                   163.0     652.1     280.7      1.4                 38->34                      9->17   512


## Recommendation

Ranked by INT8 on-disk size (the dominant constraint for an iOS app bundle), with retrieval
stability under quantization (`top1_shift`) shown so a small model that also breaks retrieval
quality doesn't look artificially attractive.

In [11]:
ranked = sorted(results, key=lambda r: r["int8_mb"])
print(f"{'Model':42s} {'INT8(MB)':>9s} {'ms/track(int8)':>15s} {'top1 shift':>10s}")
for r in ranked:
    print(
        f"{r['label']:42s} {r['int8_mb']:9.1f} {r['ms_per_track_int8']:15.0f} "
        f"{r['top1_shift']}/{len(QUERY_TEXTS):>8}"
    )

Model                                       INT8(MB)  ms/track(int8) top1 shift
CLAP (LAION, general-audio, HTSAT-unfused)     271.9              49 3/       6
Wav2CLIP + CLIP ViT-B/32 text                  280.7              34 0/       6
CLAP (LAION, music+speech-finetuned)           312.8              94 4/       6
MuQ-MuLan-large (Tencent)                     1411.6            2393 1/       6
